# **1. Importy, konfiguracja i stałe globalne**

In [1]:
import os
import glob
import random
import warnings
import numpy as np
import scipy.signal as signal
from scipy.signal import find_peaks
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import ASTFeatureExtractor, ASTModel

warnings.filterwarnings("ignore")
torch.cuda.empty_cache()

# PARAMETRY GLOBALNE
SEG_SR = 4000
CLF_SR = 16000
HOP_LENGTH_SEG = 100
MAX_LEN_SEC = 20
N_MELS_SEG = 64

# AST Klasyfikator
CLF_SEG_LEN_SEC = 5
CLF_SEG_TARGET_LEN = CLF_SR * CLF_SEG_LEN_SEC
BATCH_SIZE_CLF = 16
EPOCHS_CLF = 20
EPOCHS_SEG = 30
LR_CLF = 3e-4
WARMUP_EPOCHS = 3
GRAD_CLIP = 1.0

CLASS_NAMES = ["Normal", "Crackles", "Wheezes", "Both"]
CLASS_COLORS = {"Normal": "lightgreen", "Crackles": "orange", "Wheezes": "skyblue", "Both": "salmon"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Urządzenie: {device}")

# AUTOMATYCZNE WYKRYWANIE ŚCIEŻEK
dataset_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'audio_and_txt_files' in root:
        dataset_dir = root
        break

if dataset_dir is None:
    dataset_dir = "/content/Respiratory_Sound_Database/audio_and_txt_files/"
    save_dir = "/content/"
else:
    save_dir = "/kaggle/working/"

save_crnn_path = os.path.join(save_dir, "best_lung_crnn_model.pth")
save_ast_clf_path = os.path.join(save_dir, "best_lung_ast_model.pth")
print(f"Ścieżka do danych: {dataset_dir}")

ast_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

def encode_label(c, w):
    return int(c) * 1 + int(w) * 2

Urządzenie: cuda


Ścieżka do danych: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

# **2. Przygotowanie środowiska i podział danych**

In [2]:
txt_files = glob.glob(os.path.join(dataset_dir, '*.txt'))

if not txt_files:
    raise ValueError("Nie znaleziono plików TXT! Sprawdź czy dataset jest podpięty.")

patient_ids = sorted(list(set([os.path.basename(f).split('_')[0] for f in txt_files])))
train_patients, test_patients = train_test_split(patient_ids, test_size=0.2, random_state=42)

def get_files_for_patients(patients, d_dir):
    return [f for f in glob.glob(os.path.join(d_dir, '*.wav')) if os.path.basename(f).split('_')[0] in patients]

train_audio_files = get_files_for_patients(train_patients, dataset_dir)
test_audio_files = get_files_for_patients(test_patients, dataset_dir)
print(f"Pliki treningowe: {len(train_audio_files)} | Testowe: {len(test_audio_files)}")

Pliki treningowe: 706 | Testowe: 214


# **3. Przetwarzanie sygnału i augmentacje**

In [3]:
def butter_highpass_filter(data, cutoff=50.0, fs=4000, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, cutoff / nyq, btype='high', analog=False)
    return signal.filtfilt(b, a, data)

def safe_n_fft(segment_len, default=512):
    n = default
    while n > segment_len and n > 32:
        n //= 2
    return n

def augment_audio(segment, sr):
    if random.random() > 0.8 and len(segment) >= 256:
        n_steps = random.uniform(-1, 1)
        segment = librosa.effects.pitch_shift(y=segment, sr=sr, n_steps=n_steps, n_fft=safe_n_fft(len(segment)))
    
    if random.random() > 0.5:
        segment = segment + np.random.normal(0, 0.002, len(segment))
    
    if random.random() > 0.85 and len(segment) >= 256:
        rate = random.uniform(0.85, 1.15)
        segment = librosa.effects.time_stretch(y=segment, rate=rate)
    
    if random.random() > 0.6:
        gain = random.uniform(0.7, 1.3)
        segment = segment * gain
    
    return segment

def spec_augment(spec, num_time_masks=2, num_freq_masks=2, time_mask_max=30, freq_mask_max=20):
    spec = spec.clone()
    T, F = spec.shape
    
    for _ in range(num_time_masks):
        t = random.randint(0, time_mask_max)
        t0 = random.randint(0, max(T - t, 0))
        spec[t0:t0 + t, :] = 0.0
    
    for _ in range(num_freq_masks):
        f = random.randint(0, freq_mask_max)
        f0 = random.randint(0, max(F - f, 0))
        spec[:, f0:f0 + f] = 0.0
    
    return spec

# **4. Definicje Datasetów i DataLoaderów**

In [4]:
class ICBHISegmentationDataset(Dataset):
    def __init__(self, audio_files):
        self.audio_files = audio_files
        self.max_frames = int((MAX_LEN_SEC * SEG_SR) / HOP_LENGTH_SEG) + 1
    
    def __len__(self):
        return len(self.audio_files)
    
    def __getitem__(self, idx):
        wav_path = self.audio_files[idx]
        txt_path = wav_path.replace('.wav', '.txt')
        
        audio, _ = librosa.load(wav_path, sr=SEG_SR)
        audio = butter_highpass_filter(audio, fs=SEG_SR)
        
        target_len = MAX_LEN_SEC * SEG_SR
        if len(audio) > target_len:
            audio = audio[:target_len]
        else:
            audio = np.pad(audio, (0, target_len - len(audio)))
        
        mel = librosa.feature.melspectrogram(y=audio, sr=SEG_SR, n_mels=N_MELS_SEG, hop_length=HOP_LENGTH_SEG)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        
        label_mask = np.zeros(self.max_frames)
        gaussian_window = [0.1, 0.5, 1.0, 0.5, 0.1]
        
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        start_time = float(parts[0])
                        boundary_frame = int((start_time * SEG_SR) / HOP_LENGTH_SEG)
                        
                        for i, val in enumerate(gaussian_window):
                            lbl_idx = boundary_frame - 2 + i
                            if 0 <= lbl_idx < self.max_frames:
                                label_mask[lbl_idx] = max(label_mask[lbl_idx], val)
        
        return torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0), torch.tensor(label_mask, dtype=torch.float32).unsqueeze(1)

class ICBHIASTDataset(Dataset):
    def __init__(self, audio_files, augment=False):
        self.augment = augment
        self.segments = []
        
        for wav_path in audio_files:
            txt_path = wav_path.replace('.wav', '.txt')
            if not os.path.exists(txt_path):
                continue
            
            try:
                audio, _ = librosa.load(wav_path, sr=CLF_SR)
            except:
                continue
            
            audio = butter_highpass_filter(audio, fs=CLF_SR)
            
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) < 4:
                        continue
                    
                    s, e = int(float(parts[0]) * CLF_SR), int(float(parts[1]) * CLF_SR)
                    segment = audio[s:e]
                    
                    if len(segment) < int(CLF_SR * 0.1):
                        continue
                    
                    c, w = int(parts[2]), int(parts[3])
                    label = encode_label(c, w)
                    self.segments.append((segment, label))
        
        # Indeksy segmentów klasy Both - do mixup
        self.both_indices = [i for i, (_, lbl) in enumerate(self.segments) if lbl == 3]
    
    def __len__(self):
        return len(self.segments)
    
    def __getitem__(self, idx):
        segment, label = self.segments[idx]
        segment = segment.copy().astype(np.float32)
        
        if self.augment:
            if random.random() > 0.5:
                segment = augment_audio(segment, CLF_SR)
            
            # Mixup tylko dla klasy Both
            if label == 3 and len(self.both_indices) > 1 and random.random() > 0.4:
                other_idx = random.choice(self.both_indices)
                other_seg, _ = self.segments[other_idx]
                other_seg = other_seg.copy().astype(np.float32)
                min_len = min(len(segment), len(other_seg))
                alpha = random.uniform(0.3, 0.7)
                segment[:min_len] = alpha * segment[:min_len] + (1 - alpha) * other_seg[:min_len]
        
        if len(segment) > CLF_SEG_TARGET_LEN:
            if self.augment:
                start_idx = random.randint(0, len(segment) - CLF_SEG_TARGET_LEN)
                segment = segment[start_idx: start_idx + CLF_SEG_TARGET_LEN]
            else:
                segment = segment[:CLF_SEG_TARGET_LEN]
        elif len(segment) < CLF_SEG_TARGET_LEN:
            segment = np.pad(segment, (0, CLF_SEG_TARGET_LEN - len(segment)))
        
        inputs = ast_extractor(segment, sampling_rate=CLF_SR, padding="max_length", return_tensors="pt")
        input_values = inputs.input_values.squeeze(0)
        
        if self.augment and random.random() > 0.5:
            input_values = spec_augment(input_values)
        
        return input_values, torch.tensor(label, dtype=torch.long)

train_loader_seg = DataLoader(ICBHISegmentationDataset(train_audio_files), batch_size=16, shuffle=True)
train_dataset_clf = ICBHIASTDataset(train_audio_files, augment=True)
test_dataset_clf = ICBHIASTDataset(test_audio_files, augment=False)

train_labels_4class = [lbl for _, lbl in train_dataset_clf.segments]
class_counts = np.bincount(train_labels_4class, minlength=4)
sample_weights = [1.0 / (count + 1e-6) for count in class_counts]

# NAPRAWIONO: pętla przechodzi teraz poprawnie po train_labels_4class zamiast po segmentach
weights_per_sample = [sample_weights[lbl] for lbl in train_labels_4class]
sampler = WeightedRandomSampler(weights=weights_per_sample, num_samples=len(weights_per_sample), replacement=True)

train_loader_clf = DataLoader(train_dataset_clf, batch_size=BATCH_SIZE_CLF, sampler=sampler, num_workers=2)
test_loader_clf = DataLoader(test_dataset_clf, batch_size=BATCH_SIZE_CLF, shuffle=False, num_workers=2)

print(f"Rozkład klas treningowych: {class_counts}")
print(f"Segmenty Both do mixup: {len(train_dataset_clf.both_indices)}")

Rozkład klas treningowych: [2842 1347  662  344]
Segmenty Both do mixup: 344


# **5. Definicje architektur modeli i funkcji straty**

In [5]:
class RespiratorySegmentationCRNN(nn.Module):
    def __init__(self, input_channels=1, hidden_size=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 1))
        )
        self.rnn = nn.GRU(input_size=512, hidden_size=hidden_size, num_layers=2, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_size * 2, 1)
    
    def forward(self, x):
        x = self.cnn(x)
        b, c, m, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(b, t, c * m)
        x, _ = self.rnn(x)
        return self.classifier(x)

class LungSoundAST(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.3),
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        out = self.ast(x)
        cls_tok = out.last_hidden_state[:, 0, :]
        return self.classifier(cls_tok)

class FocalLoss(nn.Module):
    """
    Focal Loss - skupia się na trudnych przykładach (klasa Both).
    gamma=2 oznacza że łatwe przykłady dostają wagę ~0, trudne ~1.
    """
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        focal_loss = ((1 - pt) ** self.gamma) * ce
        return focal_loss.mean()

print("Architektury modeli i FocalLoss załadowane.")

Architektury modeli i FocalLoss załadowane.


# **6. Trening modelu segmentacji (CRNN)**

In [6]:
model_seg = RespiratorySegmentationCRNN().to(device)
criterion_seg = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]).to(device))
optimizer_seg = optim.Adam(model_seg.parameters(), lr=0.001)
scheduler_seg = optim.lr_scheduler.CosineAnnealingLR(optimizer_seg, T_max=EPOCHS_SEG, eta_min=1e-5)
print("START TRENINGU SEGMENTACJI")

best_f1_seg = 0.0
save_crnn_path = "/kaggle/working/best_crnn_model.pth"

for epoch in range(EPOCHS_SEG):
    model_seg.train()
    total_loss = 0
    
    for mels, labels in train_loader_seg:
        optimizer_seg.zero_grad()
        mels, labels = mels.to(device), labels.to(device)
        
        # Augmentacja mel spektrogramu w czasie treningu
        if random.random() > 0.5:
            noise = torch.randn_like(mels) * 0.1
            mels = mels + noise
        
        if random.random() > 0.5:
            gain = random.uniform(0.8, 1.2)
            mels = mels * gain
        
        loss = criterion_seg(model_seg(mels), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_seg.parameters(), 1.0)
        optimizer_seg.step()
        total_loss += loss.item()
    
    scheduler_seg.step()
    avg_loss = total_loss / len(train_loader_seg)
    current_lr = scheduler_seg.get_last_lr()[0]
    print(f"Epoka [{epoch+1:02d}/{EPOCHS_SEG}] Strata: {avg_loss:.4f} | LR: {current_lr:.6f}")

torch.save(model_seg.state_dict(), save_crnn_path)
print(f"\nModel CRNN zapisany: {save_crnn_path}")

START TRENINGU SEGMENTACJI
Epoka [01/30] Strata: 0.2749 | LR: 0.000997
Epoka [02/30] Strata: 0.2396 | LR: 0.000989
Epoka [03/30] Strata: 0.2266 | LR: 0.000976
Epoka [04/30] Strata: 0.2127 | LR: 0.000957
Epoka [05/30] Strata: 0.2056 | LR: 0.000934
Epoka [06/30] Strata: 0.1994 | LR: 0.000905
Epoka [07/30] Strata: 0.1917 | LR: 0.000873
Epoka [08/30] Strata: 0.1925 | LR: 0.000836
Epoka [09/30] Strata: 0.1825 | LR: 0.000796
Epoka [10/30] Strata: 0.1772 | LR: 0.000753
Epoka [11/30] Strata: 0.1729 | LR: 0.000706
Epoka [12/30] Strata: 0.1649 | LR: 0.000658
Epoka [13/30] Strata: 0.1610 | LR: 0.000608
Epoka [14/30] Strata: 0.1544 | LR: 0.000557
Epoka [15/30] Strata: 0.1505 | LR: 0.000505
Epoka [16/30] Strata: 0.1440 | LR: 0.000453
Epoka [17/30] Strata: 0.1399 | LR: 0.000402
Epoka [18/30] Strata: 0.1371 | LR: 0.000352
Epoka [19/30] Strata: 0.1311 | LR: 0.000304
Epoka [20/30] Strata: 0.1253 | LR: 0.000258
Epoka [21/30] Strata: 0.1193 | LR: 0.000214
Epoka [22/30] Strata: 0.1172 | LR: 0.000174
Epoka

# **#7. Trening modelu klasyfikacji (AST)**

In [7]:
counts = np.bincount(train_labels_4class, minlength=4)
print(f"Liczba próbek per klasa: Normal={counts[0]}, Crackles={counts[1]}, Wheezes={counts[2]}, Both={counts[3]}")

class_weights = 1.0 / (counts + 1e-6)
class_weights[3] *= 1.5
class_weights = class_weights / np.sum(class_weights) * 4.0
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Wagi klas: {class_weights.round(3)}")

model_clf = LungSoundAST(num_classes=4).to(device)

criterion_clf = nn.CrossEntropyLoss(weight=class_weights_tensor)
backbone_params = [p for n, p in model_clf.named_parameters() if 'classifier' not in n]
head_params = [p for n, p in model_clf.named_parameters() if 'classifier' in n]
optimizer_clf = optim.AdamW([
    {'params': backbone_params, 'lr': LR_CLF * 0.05},
    {'params': head_params, 'lr': LR_CLF}
], weight_decay=1e-2)

def get_lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS_CLF - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer_clf, lr_lambda=get_lr_lambda)

def evaluate_clf_loader(model, loader):
    model.eval()
    y_true_4, y_pred_4 = [], []
    
    with torch.no_grad():
        for mels, labels in loader:
            logits = model(mels.to(device))
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1).cpu().numpy()
            y_pred_4.extend(preds)
            y_true_4.extend(labels.numpy())
    
    y_true_4 = np.array(y_true_4)
    y_pred_4 = np.array(y_pred_4)
    cm = confusion_matrix(y_true_4, y_pred_4, labels=[0,1,2,3])
    
    #ICBHI Score
    total_correct_path = cm[1,1] + cm[2,2] + cm[3,3]
    total_path = np.sum(cm[1,:]) + np.sum(cm[2,:]) + np.sum(cm[3,:])
    SE = total_correct_path / total_path if total_path > 0 else 0
    
    SP = cm[0,0] / np.sum(cm[0,:]) if np.sum(cm[0,:]) > 0 else 0
    icbhi = 0.5 * (SE + SP) * 100
    
    return icbhi, y_true_4, y_pred_4

save_ast_path = "/kaggle/working/best_ast_model.pth"
best_icbhi = 0.0
print("\nSTART TRENINGU KLASYFIKACJI")

for epoch in range(EPOCHS_CLF):
    model_clf.train()
    total_loss = 0.0
    
    for mels, labels in train_loader_clf:
        mels, labels = mels.to(device), labels.to(device)
        optimizer_clf.zero_grad()
        loss = criterion_clf(model_clf(mels), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_clf.parameters(), GRAD_CLIP)
        optimizer_clf.step()
        total_loss += loss.item()
    
    scheduler.step()
    icbhi, _, _ = evaluate_clf_loader(model_clf, test_loader_clf)
    
    status = ""
    if icbhi > best_icbhi:
        best_icbhi = icbhi
        torch.save({'model_state_dict': model_clf.state_dict(), 'icbhi_score': best_icbhi}, save_ast_path)
        status = " ✅ Nowy najlepszy!"
    
    print(f"Epoka [{epoch+1:02d}/{EPOCHS_CLF}] Loss: {total_loss/len(train_loader_clf):.4f} | ICBHI: {icbhi:.2f}%{status}")

print(f"\nNajlepszy ICBHI Score: {best_icbhi:.2f}%")
print(f"Model zapisany: {save_ast_path}")

Liczba próbek per klasa: Normal=2842, Crackles=1347, Wheezes=662, Both=344
Wagi klas: [0.202 0.426 0.867 2.504]


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



START TRENINGU KLASYFIKACJI
Epoka [01/20] Loss: 0.8092 | ICBHI: 45.53% ✅ Nowy najlepszy!
Epoka [02/20] Loss: 0.5967 | ICBHI: 44.57%
Epoka [03/20] Loss: 0.4719 | ICBHI: 42.07%
Epoka [04/20] Loss: 0.4109 | ICBHI: 49.36% ✅ Nowy najlepszy!
Epoka [05/20] Loss: 0.3522 | ICBHI: 57.22% ✅ Nowy najlepszy!
Epoka [06/20] Loss: 0.3262 | ICBHI: 49.73%
Epoka [07/20] Loss: 0.2890 | ICBHI: 58.72% ✅ Nowy najlepszy!
Epoka [08/20] Loss: 0.2627 | ICBHI: 51.84%
Epoka [09/20] Loss: 0.2177 | ICBHI: 54.85%
Epoka [10/20] Loss: 0.2330 | ICBHI: 55.32%
Epoka [11/20] Loss: 0.1993 | ICBHI: 53.97%
Epoka [12/20] Loss: 0.1680 | ICBHI: 58.74% ✅ Nowy najlepszy!
Epoka [13/20] Loss: 0.1527 | ICBHI: 58.13%
Epoka [14/20] Loss: 0.1486 | ICBHI: 59.71% ✅ Nowy najlepszy!
Epoka [15/20] Loss: 0.1616 | ICBHI: 58.70%
Epoka [16/20] Loss: 0.1392 | ICBHI: 60.35% ✅ Nowy najlepszy!
Epoka [17/20] Loss: 0.1272 | ICBHI: 59.61%
Epoka [18/20] Loss: 0.0991 | ICBHI: 59.18%
Epoka [19/20] Loss: 0.1080 | ICBHI: 60.45% ✅ Nowy najlepszy!
Epoka [20/